# 03 · Ridge y Lasso desde cero: por qué encogen distinto

**Módulo 3 · Sesión 7** — Regresión avanzada

## Objetivos

1. Construir datos sintéticos con dos predictores casi idénticos (multicolinealidad
   extrema, a propósito) y dos predictores puro ruido, y **medir** —no solo afirmar— la
   inestabilidad que eso produce en los coeficientes de OLS.
2. Extender el `costo`/`gradiente` de `01-descenso-gradiente-intuicion.ipynb` con una
   penalización $L_2$, para implementar Ridge a mano.
3. Implementar Lasso a mano con **descenso por coordenadas** (la regla de umbral suave), y
   verificar contra `scikit-learn`.
4. Graficar la trayectoria de regularización de cada uno y comparar cómo encoge cada
   variable a medida que $\lambda$ crece.

**Paquetes:** `numpy`, `pandas`, `matplotlib`, `scikit-learn` (solo para validar).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, Ridge

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. Datos sintéticos: colinealidad extrema a propósito

`x1` y `x2` son casi la misma variable (se construyen a partir de una base común más ruido
mínimo). `x3` y `x4` son ruido puro, sin relación con `y`. El objetivo verdadero depende de
la **suma** de `x1` y `x2`, no de cada una por separado — el escenario exacto donde OLS se
desestabiliza.

In [ ]:
n = 300
base = rng.normal(0, 1, n)
x1 = base + rng.normal(0, 0.05, n)
x2 = base + rng.normal(0, 0.05, n)
x3 = rng.normal(0, 1, n)
x4 = rng.normal(0, 1, n)
X = np.column_stack([x1, x2, x3, x4])
nombres = ["x1", "x2", "x3_ruido", "x4_ruido"]

beta_verdadero = np.array([4.0, 4.0, 0.0, 0.0])
y = 3 + X @ beta_verdadero + rng.normal(0, 1, n)

print(f"correlación(x1, x2) = {np.corrcoef(x1, x2)[0, 1]:.4f}")
print(f"correlación(x1, x3_ruido) = {np.corrcoef(x1, x3)[0, 1]:.4f}")

## 2. Cuánto se desestabiliza OLS

Se ajusta OLS (ecuación normal, igual que en `01-regresion-lineal.md`) sobre 200 muestras
bootstrap de los mismos datos, y se compara la variabilidad de cada coeficiente individual
contra la variabilidad de su **suma**.

In [ ]:
def ols_normal(X, y):
    X_disenio = np.hstack([np.ones((X.shape[0], 1)), X])
    beta = np.linalg.solve(X_disenio.T @ X_disenio, X_disenio.T @ y)
    return beta[1:]  # se descarta el intercepto para este análisis


betas_bootstrap = []
for _ in range(200):
    idx = rng.integers(0, n, n)
    betas_bootstrap.append(ols_normal(X[idx], y[idx]))
betas_bootstrap = np.array(betas_bootstrap)

resumen_inestabilidad = pd.DataFrame(
    {
        "cantidad": ["beta_x1", "beta_x2", "beta_x1 + beta_x2"],
        "media": [
            betas_bootstrap[:, 0].mean(),
            betas_bootstrap[:, 1].mean(),
            (betas_bootstrap[:, 0] + betas_bootstrap[:, 1]).mean(),
        ],
        "desv_estandar": [
            betas_bootstrap[:, 0].std(),
            betas_bootstrap[:, 1].std(),
            (betas_bootstrap[:, 0] + betas_bootstrap[:, 1]).std(),
        ],
    }
)
resumen_inestabilidad.round(3)

`beta_x1` y `beta_x2` individuales tienen una desviación estándar de bootstrap enorme frente
a su valor verdadero (4 cada una) — el modelo no logra decidir consistentemente cuánto
"crédito" darle a cada una. Su **suma**, en cambio, es mucho más estable: eso es exactamente
lo que dice `03-multicolinealidad-polinomica.md` — la predicción y las combinaciones
identificables se mantienen estables; los coeficientes individuales, no.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].hist(betas_bootstrap[:, 0], bins=25, alpha=0.6, label="beta_x1")
ejes[0].hist(betas_bootstrap[:, 1], bins=25, alpha=0.6, label="beta_x2")
ejes[0].axvline(4.0, color="black", linestyle="--", label="valor verdadero")
ejes[0].set_title("Coeficientes individuales (inestables)")
ejes[0].legend()

ejes[1].hist(betas_bootstrap[:, 0] + betas_bootstrap[:, 1], bins=25, color="tab:green")
ejes[1].axvline(8.0, color="black", linestyle="--", label="suma verdadera")
ejes[1].set_title("Suma beta_x1 + beta_x2 (estable)")
ejes[1].legend()
plt.tight_layout()
plt.show()

## 3. Ridge, extendiendo el descenso del gradiente del notebook 01

La función de costo de Ridge agrega $\lambda \lVert \boldsymbol{\beta} \rVert_2^2$ —sin
penalizar el intercepto, que aquí se maneja centrando `y` y estandarizando `X`, así que el
intercepto óptimo es cero y se omite—.

In [ ]:
X_z = (X - X.mean(axis=0)) / X.std(axis=0)
y_c = y - y.mean()


def costo_ridge(beta, X, y, lam):
    return np.mean((y - X @ beta) ** 2) + lam * np.sum(beta**2)


def gradiente_ridge(beta, X, y, lam):
    return -2 * X.T @ (y - X @ beta) / len(y) + 2 * lam * beta


def ridge_descenso(X, y, lam, tasa=0.1, pasos=3000):
    beta = np.zeros(X.shape[1])
    for _ in range(pasos):
        beta = beta - tasa * gradiente_ridge(beta, X, y, lam)
    return beta

Se valida contra `sklearn.linear_model.Ridge` en un $\lambda$ fijo. `scikit-learn` penaliza
la suma de residuales al cuadrado (no el promedio), así que su `alpha` equivale a
$\lambda \times n$.

In [ ]:
lam_prueba = 0.3
beta_mio = ridge_descenso(X_z, y_c, lam_prueba)
beta_sklearn = Ridge(alpha=lam_prueba * n, fit_intercept=False).fit(X_z, y_c).coef_

pd.DataFrame(
    {"variable": nombres, "ridge_a_mano": beta_mio.round(4), "sklearn": beta_sklearn.round(4)}
)

## 4. Trayectoria de Ridge

Ridge sí tiene solución cerrada (`04-regularizacion.md`), así que calcular la trayectoria
completa —muchos valores de $\lambda$— es mucho más rápido resolviendo el sistema lineal
directamente que repitiendo el descenso para cada uno.

In [ ]:
def ridge_cerrada(X, y, lam):
    p = X.shape[1]
    return np.linalg.solve(X.T @ X + lam * len(y) * np.eye(p), X.T @ y)


lambdas = np.logspace(-2, 1.2, 25)
trayectoria_ridge = np.array([ridge_cerrada(X_z, y_c, lam) for lam in lambdas])

fig, eje = plt.subplots(figsize=(7, 4.5))
for j, nombre in enumerate(nombres):
    eje.plot(lambdas, trayectoria_ridge[:, j], marker="o", markersize=3, label=nombre)
eje.set_xscale("log")
eje.axhline(0, color="black", linewidth=0.8)
eje.set_xlabel("$\\lambda$ (escala log)")
eje.set_ylabel("Coeficiente")
eje.set_title("Trayectoria de Ridge")
eje.legend()
plt.tight_layout()
plt.show()

`beta_x1` y `beta_x2` se encogen **juntas y de forma simétrica** en todo el rango de
$\lambda$ — Ridge no elige entre ellas, las trata como un bloque. Las dos variables ruido se
acercan a cero rápido, pero nunca llegan exactamente: a $\lambda=15.8$, la mayor todavía vale
$\approx 0.013$. Ridge encoge; no selecciona.

## 5. Lasso, con descenso por coordenadas

Lasso no tiene gradiente en $\beta_j=0$, así que el descenso del gradiente ordinario no
aplica. El método estándar es **descenso por coordenadas**: se actualiza una variable a la
vez, dejando las demás fijas, con la regla de **umbral suave** (*soft-thresholding*):

$$
\beta_j \leftarrow \frac{1}{n}\text{umbral}\!\left(\mathbf{x}_j^\top \mathbf{r}_{-j},\;
\lambda\right), \qquad \text{umbral}(a, \lambda) = \text{signo}(a)\max(|a|-\lambda, 0)
$$

donde $\mathbf{r}_{-j}$ es el residual calculado **sin** la contribución de la variable $j$.
Si la correlación de $x_j$ con ese residual es menor que $\lambda$ en valor absoluto, el
umbral la manda directamente a cero — el mecanismo detrás de la selección de variables.

In [ ]:
def umbral_suave(a, lam):
    return np.sign(a) * np.maximum(np.abs(a) - lam, 0.0)


def lasso_coordenadas(X, y, lam, beta_inicial, barridos):
    beta = beta_inicial.copy()
    n = len(y)
    for _ in range(barridos):
        for j in range(X.shape[1]):
            residual_sin_j = y - X @ beta + X[:, j] * beta[j]
            rho_j = X[:, j] @ residual_sin_j / n
            beta[j] = umbral_suave(rho_j, lam)  # columnas estandarizadas: escala ya es 1
    return beta

Validación contra `scikit-learn`, con suficientes barridos para converger del todo (la
colinealidad extrema de `x1`/`x2` hace que el descenso por coordenadas tarde más en
asentarse — el mismo fenómeno de convergencia lenta de `02-descenso-gradiente.md`, aquí por
colinealidad en vez de por falta de escalado).

In [ ]:
beta_lasso_mio = lasso_coordenadas(X_z, y_c, lam_prueba, np.zeros(4), barridos=2000)
beta_lasso_sklearn = (
    Lasso(alpha=lam_prueba, fit_intercept=False, max_iter=100_000, tol=1e-10)
    .fit(X_z, y_c)
    .coef_
)

pd.DataFrame(
    {
        "variable": nombres,
        "lasso_a_mano": beta_lasso_mio.round(4),
        "sklearn": beta_lasso_sklearn.round(4),
    }
)

## 6. Trayectoria de Lasso

Se recorre la misma rejilla de $\lambda$, de mayor a menor, reutilizando la solución del
paso anterior como punto de partida del siguiente (*warm start*): cada $\lambda$ empieza
muy cerca de su óptimo, así que bastan pocos barridos por paso.

In [ ]:
trayectoria_lasso = []
beta_actual = np.zeros(4)
for lam in lambdas[::-1]:  # de mayor a menor
    beta_actual = lasso_coordenadas(X_z, y_c, lam, beta_actual, barridos=150)
    trayectoria_lasso.append(beta_actual.copy())
trayectoria_lasso = np.array(trayectoria_lasso[::-1])

fig, eje = plt.subplots(figsize=(7, 4.5))
for j, nombre in enumerate(nombres):
    eje.plot(lambdas, trayectoria_lasso[:, j], marker="o", markersize=3, label=nombre)
eje.set_xscale("log")
eje.axhline(0, color="black", linewidth=0.8)
eje.set_xlabel("$\\lambda$ (escala log)")
eje.set_ylabel("Coeficiente")
eje.set_title("Trayectoria de Lasso")
eje.legend()
plt.tight_layout()
plt.show()

La diferencia con Ridge es clara: las dos variables ruido llegan a **cero exacto** desde
$\lambda \approx 0.06$ — Lasso las elimina del modelo, no solo las encoge. `beta_x1` y
`beta_x2`, en cambio, se mantienen **juntas** shrinking en paralelo durante buena parte de la
trayectoria, y solo se anulan ambas casi al mismo $\lambda$ (entre 6 y 9), en vez de que una
desaparezca mucho antes que la otra. Lasso no logra decidir de forma limpia cuál de las dos
variables casi idénticas "vale la pena" conservar — la misma inestabilidad frente a grupos
correlacionados que `04-regularizacion.md` da como motivación para usar **Elastic Net** en
vez de Lasso puro cuando el objetivo es predictores agrupados, no aislados.

## Resumen

| Lo que se midió | Conecta con |
|---|---|
| Coeficientes de OLS individuales inestables; su suma, estable | `03-multicolinealidad-polinomica.md` |
| Ridge (a mano y con solución cerrada) coincide con `scikit-learn` | `04-regularizacion.md`, ecuación cerrada |
| Ridge encoge en bloque, sin llegar nunca a cero exacto | Geometría del círculo $L_2$ |
| Lasso (descenso por coordenadas) anula el ruido, pero no distingue entre `x1`/`x2` | Geometría del diamante $L_1$; motivación de Elastic Net |